#Swiggy RAG — Annual Report Q&A
**Stack:** PyMuPDF → sentence-transformers → FAISS + BM25 → RoBERTa extractive QA

In [ ]:
#Installation
!pip install -q PyMuPDF sentence-transformers faiss-cpu transformers torch
print('Done')

In [ ]:
#Imports
import re, math, time
import numpy as np
import fitz
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
print('Imports OK')

In [ ]:
#Config + helpers
CHUNK_SIZE  = 350
OVERLAP     = 70
TOP_K       = 5
TOP_K_QA    = 3
EMBED_MODEL = 'all-MiniLM-L6-v2'
QA_MODEL    = 'deepset/roberta-base-squad2'
STOPWORDS = {
    'the','is','in','it','of','and','to','a','an','that','this','for','are',
    'was','be','as','at','by','or','from','with','on','have','has','had',
    'not','but','what','all','were','they','we','their','can','will','its',
    'would','which','there','been','one','also','more','into','over','such',
    'our','about','when','than','then','so','if','do','up','out','no'
}
def clean_text(raw):
    t = raw.replace('\ufb01','fi').replace('\ufb02','fl').replace('\ufb00','ff')
    t = re.sub(r'(\w)-\n(\w)', r'\1\2', t)
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'(?m)^.{1,2}$', '', t)
    return t.strip()
def chunk_text(full):
    words = full.split()
    chunks, i = [], 0
    while i < len(words):
        text = ' '.join(words[i:i+CHUNK_SIZE]).strip()
        if len(text) > 80:
            chunks.append({'id': len(chunks), 'text': text})
        i += CHUNK_SIZE - OVERLAP
    return chunks
def tokenize(text):
    toks = re.sub(r'[^a-z0-9\s]', ' ', text.lower()).split()
    return [t for t in toks if len(t) > 2 and t not in STOPWORDS]
class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b, self.N = k1, b, len(corpus)
        df, self.tfs, total = {}, [], 0
        for doc in corpus:
            toks = tokenize(doc)
            total += len(toks)
            freq, seen = {}, set()
            for t in toks:
                freq[t] = freq.get(t,0)+1; seen.add(t)
            for t in seen: df[t] = df.get(t,0)+1
            self.tfs.append({'dl':len(toks),'freq':freq})
        self.avgdl = total / max(self.N,1)
        self.idf = {t: math.log((self.N-n+0.5)/(n+0.5)+1) for t,n in df.items()}
    def scores(self, query):
        s = np.zeros(self.N, dtype=np.float32)
        for term in tokenize(query):
            idf = self.idf.get(term, 0.0)
            if not idf: continue
            for i, e in enumerate(self.tfs):
                tf = e['freq'].get(term, 0)
                if not tf: continue
                s[i] += idf*tf*(self.k1+1)/(tf+self.k1*(1-self.b+self.b*e['dl']/self.avgdl))
        return s
print('Helpers ready')

In [ ]:
#Upload PDF
print('Upload the PDF')
uploaded = files.upload()

In [ ]:
#Index
fname    = list(uploaded.keys())[0]
pdf_data = uploaded[fname]
print(f'{fname}  ({len(pdf_data)/1024/1024:.1f} MB)\n')
print('1/4 Extracting text…')
doc = fitz.open(stream=pdf_data, filetype='pdf')
full_text   = clean_text('\n'.join(p.get_text('text') for p in doc))
total_pages = len(doc)
print(f'     {total_pages} pages · {len(full_text.split()):,} words')

print('2/4 Chunking…')
chunks = chunk_text(full_text)
print(f'     {len(chunks)} chunks')

print('3/4 Embedding + FAISS index…')
t0 = time.time()
import torch
device = 0 if torch.cuda.is_available() else -1
embedder = SentenceTransformer(EMBED_MODEL)
vecs = embedder.encode(
    [c['text'] for c in chunks],
    batch_size=128, normalize_embeddings=True, show_progress_bar=True
).astype(np.float32)
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
bm25 = BM25([c['text'] for c in chunks])

print('4/4 Loading QA model…')
qa_pipe = pipeline('question-answering', model=QA_MODEL, device=device)

print(f'\n Ready in {round(time.time()-t0,1)}s  |  GPU: {torch.cuda.is_available()}')

In [ ]:
#Interactive Q&A widget

def retrieve(query, top_k=TOP_K):
    q_vec = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    cos_s, idxs = index.search(q_vec, min(top_k*4, len(chunks)))
    bm25_s = bm25.scores(query)
    mx = bm25_s.max() or 1.0
    results = []
    for idx, cos in zip(idxs[0], cos_s[0]):
        if idx < 0: continue
        h = 0.65*float(cos) + 0.35*float(bm25_s[idx]/mx)
        results.append({**chunks[idx], 'score':round(h,4), 'cos':round(float(cos),4)})
    return sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]

def ask(query):
    sources = retrieve(query)
    if not sources:
        return 'No relevant content found.', 0.0, []
    context = ' '.join(s['text'] for s in sources[:TOP_K_QA])
    res  = qa_pipe(question=query, context=context, max_answer_len=200)
    ans  = res['answer'].strip()
    conf = round(res['score'], 4)
    if conf < 0.05 or len(ans) < 3:
        ans  = 'The report does not contain a clear answer. Check source chunks below.'
        conf = 0.0
    return ans, conf, sources

CSS = """
<style>
.rbox{background:#111;border:1px solid #222;border-radius:10px;padding:18px 22px;
      margin:8px 0;font-family:'JetBrains Mono',monospace;}
.rq{color:#fc8019;font-size:13px;font-weight:bold;margin-bottom:10px;}
.ra{color:#ede9e3;font-size:14px;line-height:1.75;}
.rc{color:#666;font-size:11px;margin-top:8px;}
.rl{color:#444;font-size:10px;letter-spacing:1px;text-transform:uppercase;margin:12px 0 5px;}
.rchunk{background:#1a1a1a;border-left:3px solid #fc8019;padding:8px 12px;
        margin:4px 0;border-radius:0 6px 6px 0;font-size:11px;color:#888;line-height:1.5;}
.rch{color:#fc8019;font-size:9px;letter-spacing:1px;text-transform:uppercase;margin-bottom:3px;}
</style>
"""

SUGGESTED = [
    "What was Swiggy's total revenue for FY 2023-24?",
    "How many active restaurants are on the platform?",
    "What is Swiggy Instamart and how did it perform?",
    "What are the key risk factors mentioned in the report?",
    "Who are the board of directors?",
    "What is the adjusted EBITDA or operating loss?",
    "What is Swiggy's total gross order value (GOV)?",
    "What new products or initiatives did Swiggy launch?",
]

def run_query(q):
    with out:
        clear_output(wait=True)
        display(HTML(CSS+f'<div class="rbox"><div class="rq">\u2753 {q}</div><div class="ra">\u23f3 Thinking\u2026</div></div>'))
    ans, conf, srcs = ask(q)
    pct  = int(conf*100)
    bar  = '\u2588'*(pct//5) + '\u2591'*(20-pct//5)
    schunks = ''.join(
        f'<div class="rchunk"><div class="rch">Chunk #{s["id"]} &middot; hybrid {s["score"]} &middot; cos {s["cos"]}</div>'
        f'{s["text"][:260]}\u2026</div>' for s in srcs
    )
    html = f'''{CSS}<div class="rbox">
      <div class="rq">\u2753 {q}</div>
      <div class="ra">{ans}</div>
      <div class="rc">{bar} confidence: {pct}%</div>
      <div class="rl">Retrieved chunks</div>{schunks}
    </div>'''
    with out:
        clear_output(wait=True)
        display(HTML(html))

txt  = widgets.Text(placeholder='Ask anything about the Annual Report…',
                    layout=widgets.Layout(width='80%'))
btn  = widgets.Button(description='Ask', button_style='warning',
                      layout=widgets.Layout(width='18%'))
out  = widgets.Output()
sbtns = [widgets.Button(
             description=(q[:72]+'\u2026') if len(q)>73 else q,
             layout=widgets.Layout(width='100%', margin='2px 0'),
             style={'button_color':'#1c1c1c'})
         for q in SUGGESTED]

btn.on_click(lambda b: run_query(txt.value.strip()) if txt.value.strip() else None)
txt.on_submit(lambda x: run_query(txt.value.strip()) if txt.value.strip() else None)
for b, q in zip(sbtns, SUGGESTED):
    b.on_click(lambda ev, qq=q: run_query(qq))

display(HTML('<h3 style="color:#fc8019;font-family:monospace;margin-bottom:8px">Swiggy RAG \u2014 Ask the Annual Report</h3>'))
display(widgets.HBox([txt, btn]))
display(HTML('<p style="color:#555;font-size:11px;font-family:monospace;margin:6px 0 4px">\u2b07 Suggested questions:</p>'))
display(widgets.VBox(sbtns))
display(out)